# CVE-to-My-Stack Translator — Starter Notebook

CyberHack 2026 · Project 1

Use this notebook to explore each pipeline stage before or alongside `translate.py`.

**Prerequisites:** datasets in `data/` (run `python scripts/download_datasets.py` if needed).

In [ ]:
# Run from project root (Hackathon2026/)
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent  # if launched from a subfolder
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
print("Project root:", ROOT)

## Step 1 — Dataset status & smoke test

In [ ]:
from src.paths import dataset_status, resolve_nvd_path, find_kev_json, find_epss_csv
from src.loaders import smoke_test

for key, value in dataset_status().items():
    print(f"{key}: {value}")

print()
smoke_test()

## Step 1b — Explore one NVD CVE record

FKIE feeds use `cve_items[]` with `id`, `metrics`, `configurations`, `descriptions`.

In [ ]:
import json
from src.loaders import load_nvd
from src.match import extract_cpe_keys, get_cvss_score, get_description

nvd_items = load_nvd()
sample = next(item for item in nvd_items if item.get("configurations"))

print("CVE ID:", sample["id"])
print("CVSS:", get_cvss_score(sample))
print("CPE keys:", list(extract_cpe_keys(sample))[:5])
print("Description:", get_description(sample)[:200], "...")

## Step 2 — Parse sample asset list

In [ ]:
from src.loaders import parse_asset_list

asset_path = DATA_DIR / "sample_asset_list.txt"
assets = parse_asset_list(asset_path)

for a in assets:
    print(f"{a['name']!r} | version={a['version']!r}")

## Steps 3–4 — Normalisation

Edit `config/normalisation_map.json` if any asset shows as unmapped.

In [ ]:
from src.normalise import normalise_assets, load_normalisation_map

products = load_normalisation_map()
print(f"Dictionary entries: {len(products)}")

mapped, unmapped = normalise_assets(assets)
for m in mapped:
    print(f"OK  {m['user_label']} -> {m['cpe_key']} ({m['match_method']})")
for u in unmapped:
    print(f"??  {u['user_label']}")

## Steps 5–8 — Match, enrich, rank

Full NVD scan takes ~5–10 seconds on CVE-2026.json.

In [ ]:
from src.loaders import load_kev, load_epss
from src.match import match_cves_to_assets
from src.rank import enrich_and_rank
from src.summarise import add_summaries
import pandas as pd

kev_ids = load_kev()
epss_by_cve = load_epss()

raw_matches = match_cves_to_assets(nvd_items, mapped)
print(f"Raw matches: {len(raw_matches)}")

rows = enrich_and_rank(raw_matches, epss_by_cve, kev_ids, max_rows=50)
rows = add_summaries(rows)

df = pd.DataFrame(rows)
display_cols = ["cve_id", "affected_asset", "cvss", "epss", "kev", "risk_summary"]
df[display_cols].head(20)

## Step 10 — Export CSV

In [ ]:
from src.export import write_csv, print_table

out_path = OUTPUT_DIR / "prioritised_cves.csv"
write_csv(rows, out_path)
print(f"Wrote {len(rows)} rows to {out_path}")
print_table(rows, limit=15)

## Optional — Full CLI run

Same pipeline as the notebook, from the terminal:

```powershell
python translate.py data/sample_asset_list.txt
```

In [ ]:
# Uncomment to run in-notebook via subprocess:
# !python translate.py data/sample_asset_list.txt

## Demo talking points (limitations)

1. **Normalisation** — wrong alias = CVE silently missing from output.
2. **EPSS** — predictive score, not proof of safety.
3. **KEV** — absence does not mean unexploited.
4. **Versions** — MVP matches `vendor:product` only, not version ranges.